**RNNs, GRUs, and LSTMs from Scratch**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

**One Layer RNN:**

In [2]:
# implementing the default one for comparison
torch.manual_seed(1)
rnn_layer = nn.RNN(input_size=5, hidden_size=2, num_layers=1, batch_first=True)

In [3]:
w_xh = rnn_layer.weight_ih_l0
w_hh = rnn_layer.weight_hh_l0
b_xh = rnn_layer.bias_ih_l0
b_hh = rnn_layer.bias_hh_l0

In [4]:
print('W_xh shape: ', w_xh.shape)
print('W_hh shape: ', w_hh.shape)
print('b_xh shape: ', b_xh.shape)
print('b_hh shape: ', b_hh.shape)

W_xh shape:  torch.Size([2, 5])
W_hh shape:  torch.Size([2, 2])
b_xh shape:  torch.Size([2])
b_hh shape:  torch.Size([2])


the input shape for this layer is: (batch_size, seq_length, 5): the first dimension is the batch dimension (as we set batch_first to true). the second dimension corresponds to the sequence, and the last dimension corresponds to the features. Now, we call a forward pass on the rnn and manually compute the outputs at each time step and then compare them

In [6]:
x_seq = torch.tensor([[1.0]*5,[2.0]*5,[3.0]*5]).float()
x_seq = x_seq.unsqueeze(0)
x_seq.shape

torch.Size([1, 3, 5])

In [7]:
# now run that through the rnn
output, hn = rnn_layer(x_seq)
print(output.shape, hn.shape)

torch.Size([1, 3, 2]) torch.Size([1, 1, 2])


Now manually computing the single layer rnn output

In [10]:
a = torch.randn(1,2)
b = torch.zeros(a.shape)

In [11]:
b.shape

torch.Size([1, 2])

In [14]:
manual_output = []
seq_length = 3

with torch.no_grad():
    for i in range(seq_length):
        # compute the input to hidden part
        ih = (x_seq[:, i, :] @ w_xh.T) + b_xh
        if i == 0:
            hh = torch.zeros(ih.shape) @ w_hh.T + b_hh
        else:
            hh = manual_output[i-1] @ w_hh.T + b_hh
        ht = F.tanh(ih + hh)
        manual_output.append(ht)

In [24]:
torch.cat(manual_output, dim=0)

tensor([[-0.3520,  0.5253],
        [-0.6842,  0.7607],
        [-0.8649,  0.9047]])

In [25]:
output

tensor([[[-0.3520,  0.5253],
         [-0.6842,  0.7607],
         [-0.8649,  0.9047]]], grad_fn=<TransposeBackward1>)

**Successfully reproduced**

**Now onto a 2-layer RNN:**

In [26]:
torch.manual_seed(1)
rnn_layer2 = nn.RNN(input_size=5, hidden_size=2, num_layers=2, batch_first=True)

Extracting the rnn's initial initialised weights and biases:

In [35]:
w_xh1 = rnn_layer2.weight_ih_l0
w_xh2 = rnn_layer2.weight_ih_l1
w_hh1 = rnn_layer2.weight_hh_l0
w_hh2 = rnn_layer2.weight_hh_l1
b_xh1 = rnn_layer2.bias_ih_l0
b_xh2 = rnn_layer2.bias_ih_l1
b_hh1 = rnn_layer2.bias_hh_l0
b_hh2 = rnn_layer2.bias_hh_l1

In [30]:
output2, hn2 = rnn_layer2(x_seq)

In [32]:
output2.shape, hn2.shape

(torch.Size([1, 3, 2]), torch.Size([2, 1, 2]))

In [33]:
output2

tensor([[[-0.3555,  0.5987],
         [-0.5822,  0.6520],
         [-0.6262,  0.6740]]], grad_fn=<TransposeBackward1>)

In [34]:
hn2

tensor([[[-0.8649,  0.9047]],

        [[-0.6262,  0.6740]]], grad_fn=<StackBackward0>)

now the manual 2 layer rnn:

In [36]:
layers = 2
seq_length = 3

manual_out_l1 = []
manual_out_l2 = []

for l in range(layers):
    for t in range(seq_length):
        if l == 0:
            ih = (x_seq[:, t, :] @ w_xh1.T) + b_xh1
            hh = (manual_out_l1[t-1] @ w_hh1.T) + b_hh1 if t > 0 else (torch.zeros(ih.shape) @ w_hh1.T) + b_hh1
            manual_out_l1.append(F.tanh(ih + hh))
        else:
            ih = (manual_out_l1[t] @ w_xh2.T) + b_xh2
            hh = (manual_out_l2[t-1] @ w_hh2.T) + b_hh2 if t > 0 else (torch.zeros(ih.shape) @ w_hh2.T) + b_hh2
            manual_out_l2.append(F.tanh(ih + hh))

In [37]:
output2

tensor([[[-0.3555,  0.5987],
         [-0.5822,  0.6520],
         [-0.6262,  0.6740]]], grad_fn=<TransposeBackward1>)

In [38]:
torch.cat(manual_out_l2)

tensor([[-0.3555,  0.5987],
        [-0.5822,  0.6520],
        [-0.6262,  0.6740]], grad_fn=<CatBackward0>)

**Custom RNN with Layer Normalization:**

making the basic rnn manual code cleaner by using linesr layers instead of doing the matrix multiplication manually.

In [39]:
w_xh.shape

torch.Size([2, 5])

In [40]:
nn.Linear(5, 2).weight.shape

torch.Size([2, 5])

In [41]:
input_hidden = nn.Linear(5, 2)
hidden_hidden = nn.Linear(2, 2)

# for the results between the default rnn and this manual one to be comparable, we need to fix the weights  with the default rnn ones
input_hidden.weight.data = w_xh
input_hidden.bias.data = b_xh
hidden_hidden.weight.data = w_hh
hidden_hidden.bias.data = b_hh

In [42]:
out_man = []

with torch.no_grad():
    for t in range(3):
        xt = x_seq[:, t, :]
        ht = input_hidden(xt)

        if t>0:
            prev_out = out_man[t-1]
        else:
            prev_out = torch.zeros((ht.shape))
        
        ot = ht + hidden_hidden(prev_out)
        out_man.append(F.tanh(ot))

In [43]:
out_man

[tensor([[-0.3520,  0.5253]]),
 tensor([[-0.6842,  0.7607]]),
 tensor([[-0.8649,  0.9047]])]

In [48]:
out_man = torch.stack(out_man, dim=1)

In [49]:
for i in range(3):
    print(f"rnn layer output: {output[:, i, :]}")
    print(f"manual output: {out_man[:, i, :]}")

rnn layer output: tensor([[-0.3520,  0.5253]], grad_fn=<SelectBackward0>)
manual output: tensor([[-0.3520,  0.5253]])
rnn layer output: tensor([[-0.6842,  0.7607]], grad_fn=<SelectBackward0>)
manual output: tensor([[-0.6842,  0.7607]])
rnn layer output: tensor([[-0.8649,  0.9047]], grad_fn=<SelectBackward0>)
manual output: tensor([[-0.8649,  0.9047]])


Based on the above, we can easily create a custom rnn layer.

Making the code cleaner again now using a custom rnn cell

In [50]:
rnn_cell = nn.RNNCell(5,2) # qan rnn cell with an input size of 5 and an output size of 2

In [51]:
rnn_cell.weight_ih.data = w_xh
rnn_cell.weight_hh.data = w_hh
rnn_cell.bias_ih = b_xh
rnn_cell.bias_hh = b_hh

In [ ]:
out_man2 = []

for t in range(3):
    xt = x_seq[:, t, :] # get the input at a particular timestep

    if t > 0:
        hidden = rnn_cell(xt, out_man2[t-1])
    else:
        hidden = rnn_cell(xt) # default for not providing a hidden state is a tensor of zeros, so we do not need to specify anything here then
    
    out_man2.append(hidden)

In [53]:
output

tensor([[[-0.3520,  0.5253],
         [-0.6842,  0.7607],
         [-0.8649,  0.9047]]], grad_fn=<TransposeBackward1>)

In [54]:
torch.stack(out_man2, dim=1)

tensor([[[-0.3520,  0.5253],
         [-0.6842,  0.7607],
         [-0.8649,  0.9047]]], grad_fn=<StackBackward0>)

**Adding LayerNormalization to the RNN:**

In [55]:
ln = nn.LayerNorm(2) # normalize for each example, across this dimension 2 (for each timestep)
ln_out = []

for t in range(3):
    xt = x_seq[:, t, :] # get the input at that timestep

    if t > 0:
        hidden = rnn_cell(xt, ln_out[t-1])
        normalized_hidden = ln(hidden)
    else:
        hidden = rnn_cell(xt)
        normalized_hidden = ln(hidden)
    ln_out.append(normalized_hidden)

In [56]:
torch.stack(ln_out, dim=1)

tensor([[[-1.0000,  1.0000],
         [-1.0000,  1.0000],
         [-1.0000,  1.0000]]], grad_fn=<StackBackward0>)

Sometimes its recommended to normalize before calling the activation

In [66]:
class CustomRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_hidden = nn.Linear(input_size, hidden_size)
        self.hidden_hidden = nn.Linear(hidden_size, hidden_size)
        self.ln = nn.LayerNorm(hidden_size)
    
    def forward(self, x, hidden):
        out = self.input_hidden(x) + self.hidden_hidden(hidden)
        out = self.ln(out)
        return F.tanh(out)

In [67]:
cust_rnn = CustomRNNCell(5,2)

In [68]:
cust_rnn.input_hidden.weight.data = w_xh
cust_rnn.input_hidden.bias.data = b_xh
cust_rnn.hidden_hidden.weight.data = w_hh
cust_rnn.hidden_hidden.bias.data = b_hh

In [69]:
out_man3 = []

for t in range(3):
    xt = x_seq[:, t, :]

    if t > 0:
        hidden = cust_rnn(xt, out_man3[t-1])
    else:
        initial_hidden = torch.zeros((1,2))
        hidden = cust_rnn(xt, initial_hidden)
    out_man3.append(hidden)

In [70]:
torch.stack(out_man3, dim=1)

tensor([[[-0.7616,  0.7616],
         [-0.7616,  0.7616],
         [-0.7616,  0.7616]]], grad_fn=<StackBackward0>)

**LSTMs:**
- fine grained operation of the LSTM including layer normalization

In [71]:
# writing out  default lstm layer for comparison
lstm_layer = nn.LSTM(input_size=5, hidden_size=2, num_layers=1, batch_first=True)

In [72]:
# Inspecting the LSTM's variables - weights and biases
wi = lstm_layer.weight_ih_l0 # translates to these weights: (W_ii|W_if|W_ig|W_io)
wh = lstm_layer.weight_hh_l0 # trnslates to these weights: (W_hi|W_hf|W_hg|W_ho)
bi = lstm_layer.bias_ih_l0 # translates to (b_ii|b_if|b_ig|b_io)
bh = lstm_layer.bias_hh_l0 # translates to (b_hi|b_hf|b_hg|b_ho)

In [74]:
wi.shape, wh.shape, bi.shape, bh.shape

(torch.Size([8, 5]), torch.Size([8, 2]), torch.Size([8]), torch.Size([8]))

weights and biases make perfect sense, considering the defined layer shapes

In [75]:
x_seq = torch.tensor([[1.0]*5, [2.0]*5, [3.0]*5]).unsqueeze(0).float()
x_seq.shape

torch.Size([1, 3, 5])

In [76]:
output, (hn, cn) = lstm_layer(x_seq)
print(output.shape, hn.shape, cn.shape)

torch.Size([1, 3, 2]) torch.Size([1, 1, 2]) torch.Size([1, 1, 2])


Working the lstm output manually:

In [77]:
# first, lets break down the weights into what they should actually be

# below, we carefully extract the weights for the input gate, forget gate, input at particular timestamp, and output gate
wii, wif, wig, wio = wi[:2, :], wi[2:4, :], wi[4:6, :], wi[6:, :]
whi, whf, whg, who = wh[:2, :], wh[2:4, :], wh[4:6, :], wh[6:, :]
bii, bif, big, bio = bi[:2], bi[2:4], bi[4:6], bi[6:]
bhi, bhf, bhg, bho = bh[:2], bh[2:4], bh[4:6], bh[6:]

In [78]:
long_state = []
short_state = []

# loop over the sequence length: unroll the sequence through time
for t in range(3):
    # for each timestep, get the input that corresponds to that timestep
    xt = x_seq[:, t, :]

    # making all computations that do not need any form of hidden state
    # firstly, the first part of the input to the input gate
    it1 = (xt @ wii.T) + bii
    # first part of the forget gate
    ft1 = (xt @ wif.T) + bif
    # first part of the new signal processed: g
    gt1 = (xt @ wig.T) + big
    # first part of the output gate
    ot1 = (xt @ wio.T) + bio

    if t > 0:
        prev_h = short_state[t - 1]
        prev_c = long_state[t - 1]
    else:
        # otherwise, start by initialising with zeros
        prev_h = torch.zeros((xt.shape[0], 2)) # at timestep 0, we use zeros of the shape we're trying to project our inputs to
        prev_c = torch.zeros((xt.shape[0], 2))
    
    # finish up the input gate
    it = F.sigmoid(it1 + ((prev_h @ whi.T) + bhi))

    # finish up the forget gate
    ft = F.sigmoid(ft1 + ((prev_h @ whf.T) + bhf))

    # finish up the candidate signal
    gt = F.tanh(gt1 + ((prev_h @ whg.T) + bhg))

    # finish up the output gate
    ot = F.sigmoid(ot1 + ((prev_h @ who.T) + bho))

    long_state_t = (ft * prev_c) + (it * gt)
    short_state_t = ot * (F.tanh(long_state_t))

    long_state.append(long_state_t)
    short_state.append(short_state_t)

In [79]:
hn, cn

(tensor([[[-0.0642,  0.0058]]], grad_fn=<StackBackward0>),
 tensor([[[-1.2854,  0.0799]]], grad_fn=<StackBackward0>))

In [80]:
short_state[-1], long_state[-1]

(tensor([[-0.0642,  0.0058]], grad_fn=<MulBackward0>),
 tensor([[-1.2854,  0.0799]], grad_fn=<AddBackward0>))

In [81]:
output

tensor([[[-0.0059,  0.0857],
         [-0.0730,  0.0289],
         [-0.0642,  0.0058]]], grad_fn=<TransposeBackward0>)

In [82]:
torch.stack(short_state, dim=1)

tensor([[[-0.0059,  0.0857],
         [-0.0730,  0.0289],
         [-0.0642,  0.0058]]], grad_fn=<StackBackward0>)

Code above is accurate, but a bit too crude.
- what if we had our own LSTM cell 

In [83]:
lstm_cell = nn.LSTMCell(5, 2)

Initialize the weights to the lstm layer's weights

In [84]:
lstm_cell.weight_ih.data = wi
lstm_cell.weight_hh.data = wh
lstm_cell.bias_ih.data = bi
lstm_cell.bias_hh.data = bh

compute the output for the sequence, using this lstm cell

In [85]:
long_state = []
short_state = []

for t in range(3):
    xt = x_seq[:, t, :]
    if t > 0:
        prev_h = short_state[t - 1]
        prev_c = long_state[t - 1]
    else:
        prev_h = torch.zeros((xt.shape[0], 2))
        prev_c = torch.zeros((xt.shape[0], 2))
    
    short_state_t, long_state_t = lstm_cell(xt, (prev_h, prev_c))

    long_state.append(long_state_t)
    short_state.append(short_state_t)

In [86]:
hn, cn

(tensor([[[-0.0642,  0.0058]]], grad_fn=<StackBackward0>),
 tensor([[[-1.2854,  0.0799]]], grad_fn=<StackBackward0>))

In [87]:
short_state[-1], long_state[-1]

(tensor([[-0.0642,  0.0058]], grad_fn=<MulBackward0>),
 tensor([[-1.2854,  0.0799]], grad_fn=<AddBackward0>))

Clearly,  using an lstm cell along is more elegant and cleaner. however, if we were to use layer normalization, there would be no way of having the layer normalization layer in before the final activation functions - hence the need for a custom cell

In [88]:
class CustomLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input = nn.Linear(input_size, 4*hidden_size)
        self.hidden = nn.Linear(hidden_size, 4*hidden_size)
    
    def forward(self, x, prev_h, prev_c):
        computed_inputs = self.input(x)
        computed_hiddens = self.hidden(prev_h)

        # compute the input gate
        input_gate = torch.sigmoid(computed_inputs[:, :2] + computed_hiddens[:, :2])
        forget_gate = torch.sigmoid(computed_inputs[:, 2:4] + computed_hiddens[:, 2:4])
        signal = torch.tanh(computed_inputs[:, 4:6] + computed_hiddens[:, 4:6])
        output_gate = torch.sigmoid(computed_inputs[:, 6:] + computed_hiddens[:, 6:])
        long_state = forget_gate * prev_c + input_gate*signal
        return output_gate * torch.tanh(long_state), long_state

custom lstm cell with layer normalization

In [90]:
class CustomLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input = nn.Linear(input_size, 4*hidden_size)
        self.hidden = nn.Linear(hidden_size, 4*hidden_size)
        self.hidden_size = hidden_size
        self.ln = nn.LayerNorm(hidden_size)
    
    def forward(self, x, prev_h, prev_c):
        computed_inputs = self.input(x)
        computed_hiddens = self.hidden(prev_h)
        # compute the input gate
        input_gate = torch.sigmoid(computed_inputs[:, :self.hidden_size] + computed_hiddens[:, :self.hidden_size])
        forget_gate = torch.sigmoid(computed_inputs[:, self.hidden_size:2*self.hidden_size] + computed_hiddens[:, self.hidden_size:2*self.hidden_size])
        signal = torch.tanh(computed_inputs[:, 2*self.hidden_size:3*self.hidden_size] + computed_hiddens[:, 2*self.hidden_size:3*self.hidden_size])
        output_gate = torch.sigmoid(computed_inputs[:,3*self.hidden_size:4*self.hidden_size]+computed_hiddens[:,3*self.hidden_size:4*self.hidden_size])
        long_state = forget_gate * prev_c + input_gate * signal
        return output_gate * F.tanh(self.ln(long_state)), long_state

**GRU:**
- GRU from scratch implementation
- Custom GRU cell with layer normalization

In [95]:
torch.manual_seed(1)
gru_layer = nn.GRU(input_size=5, hidden_size=2, num_layers=1, bias=True, batch_first=True)

In [96]:
output, hn = gru_layer(x_seq)

In [97]:
output

tensor([[[-0.6458,  0.1718],
         [-0.8509,  0.2851],
         [-0.9287,  0.3488]]], grad_fn=<TransposeBackward1>)

In [98]:
hn

tensor([[[-0.9287,  0.3488]]], grad_fn=<StackBackward0>)

In [99]:
wi = gru_layer.weight_ih_l0 # is of representation (W_ir|W_iz|W_in)
wh = gru_layer.weight_hh_l0
bi = gru_layer.bias_ih_l0
bh = gru_layer.bias_hh_l0

In [101]:
wh.shape

torch.Size([6, 2])

Replicating the above results with a custom GRU Cell

In [102]:
class CustomGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_hidden = nn.Linear(input_size, 3*hidden_size) # 3 because of weights needed for remember gate, forget gate and the raw signal
        self.hidden_hidden = nn.Linear(hidden_size, 3*hidden_size)

        self.input_hidden.weight.data = wi
        self.input_hidden.bias.data = bi
        self.hidden_hidden.weight.data = wh
        self.hidden_hidden.bias.data = bh
        self.hidden_size = hidden_size
    
    def forward(self, x, h):
        computed_inputs = self.input_hidden(x)
        computed_hiddens = self.hidden_hidden(h)

        # r gate
        r = F.sigmoid(computed_inputs[:, :self.hidden_size] + computed_hiddens[:, :self.hidden_size])
        z = F.sigmoid(computed_inputs[:, self.hidden_size: 2*self.hidden_size] + computed_hiddens[:, self.hidden_size: 2*self.hidden_size])
        n = F.tanh(computed_inputs[:, 2*self.hidden_size:3*self.hidden_size] + r * computed_hiddens[:, 2*self.hidden_size:3*self.hidden_size])

        return (1 - z) * n + z*h

In [103]:
gru_cell = CustomGRUCell(5, 2)

In [104]:
state = []

for t in range(3):
    xt = x_seq[:, t, :]

    if t > 0:
        prev_h = state[t - 1]
    else:
        prev_h = torch.zeros((xt.shape[0], 2))
    
    o = gru_cell(xt, prev_h)

    state.append(o)

In [107]:
torch.stack(state, dim=1)

tensor([[[-0.6458,  0.1718],
         [-0.8509,  0.2851],
         [-0.9287,  0.3488]]], grad_fn=<StackBackward0>)

In [106]:
output

tensor([[[-0.6458,  0.1718],
         [-0.8509,  0.2851],
         [-0.9287,  0.3488]]], grad_fn=<TransposeBackward1>)

**successfully replicated the original layer's results**

applying layer norm to the custom GRU cell....

In [108]:
class CustomGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input = nn.Linear(input_size, 3*hidden_size)
        self.hidden = nn.Linear(hidden_size, 3*hidden_size)
        self.hidden_size = hidden_size

        # one layer norm per gate
        self.ln_r = nn.LayerNorm(hidden_size)
        self.ln_z = nn.LayerNorm(hidden_size)
        self.ln_n = nn.LayerNorm(hidden_size)
    
    def forward(self, x, prev_h):
        computed_inputs = self.input(x)
        computed_hiddens = self.hidden(prev_h)

        # gates 
        pre_r = computed_inputs[:, :self.hidden_size] + computed_hiddens[:, :self.hidden_size]
        rgate = F.sigmoid(self.ln_r(pre_r))

        pre_z = computed_inputs[:, self.hidden_size:2*self.hidden_size] + computed_hiddens[:, self.hidden_size:2*self.hidden_size]
        zgate = F.sigmoid(self.ln_z(pre_z))

        pre_n = computed_inputs[:, 2*self.hidden_size:] + (rgate * computed_hiddens[:, 2*self.hidden_size:])
        ngate = F.tanh(self.ln_n(pre_n))

        h = (1 - zgate)*n_gate + zgate*prev_h
        return h